# UFC Fight Outcome — EDA on the Cleaned Dataset\n\nThis runs the pipeline built in `src/data/` and `src/features/` (clean -> engineer -> split) and explores the result: target balance, feature distributions, relationship of each feature to the outcome, remaining missingness, and whether train/val/test drift from each other now that the split is chronological.\n\nSource tables: `data/master.csv` (fight-level) and `data/round.csv` (round-level)."

In [ ]:
import sys
sys.path.insert(0, "../src/data")
sys.path.insert(0, "../src/features")

import pandas as pd
import matplotlib.pyplot as plt

from cleaning import clean_fights, clean_rounds
from engineering import add_matchup_features, add_fighter_history_features, add_early_pace_features
from splitting import split_fights

master = pd.read_csv("../data/master.csv")
rounds = clean_rounds(pd.read_csv("../data/round.csv"))

fights = clean_fights(master)
fights = add_matchup_features(fights)
fights = add_fighter_history_features(fights)
fights = add_early_pace_features(fights, rounds)

fights["winner_side"] = "red"
fights.loc[fights["winner_id"] == fights["b_fighter_id"], "winner_side"] = "blue"
fights["is_red_win"] = (fights["winner_side"] == "red").astype(int)

train, val, test = split_fights(fights)

fights.shape, train.shape, val.shape, test.shape

: 

## Target Balance\n\nEarlier full-dataset EDA found red wins ~68% overall, driven almost entirely by a pre-2010 recording artifact (see `docs/results.md`). Checking it again here on the cleaned data, split out by train/val/test, confirms whether that's still true after the temporal split."

In [ ]:
for name, part in [("train", train), ("val", val), ("test", test)]:
    rate = part["is_red_win"].mean()
    print(f"{name:6s} n={len(part):5d}  red_win_rate={rate:.3f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["train", "val", "test"], [train["is_red_win"].mean(), val["is_red_win"].mean(), test["is_red_win"].mean()])
ax.axhline(0.5, color="gray", linestyle="--", label="coin flip")
ax.set_ylabel("red win rate")
ax.legend()
plt.show()

## Feature Distributions\n\nThe engineered features from `src/features/engineering.py`: physical/age differences and each fighter's historical record coming into the fight."

In [ ]:
feature_cols = ["height_diff", "reach_diff", "weight_diff", "age_diff",
                "experience_diff", "prior_win_rate_diff",
                "r_days_since_last_fight", "b_days_since_last_fight"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    axes[i].hist(fights[col].dropna(), bins=40)
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

fights[feature_cols].describe().T

## Feature Relationship With the Outcome\n\nAll diff features are defined as red minus blue, so a positive value means red has the edge. If these features carry real signal, red winning should skew each distribution positive; if a feature shows no separation at all, it's not worth including."

In [ ]:
diff_cols = ["height_diff", "reach_diff", "age_diff", "experience_diff", "prior_win_rate_diff"]

print(fights.groupby("winner_side")[diff_cols].mean())

fig, axes = plt.subplots(1, len(diff_cols), figsize=(18, 4))
for i, col in enumerate(diff_cols):
    data = [fights.loc[fights["winner_side"] == "red", col].dropna(),
            fights.loc[fights["winner_side"] == "blue", col].dropna()]
    axes[i].boxplot(data, tick_labels=["red won", "blue won"])
    axes[i].set_title(col)
    axes[i].axhline(0, color="gray", linestyle="--")
plt.tight_layout()
plt.show()

print("\ncorrelation with is_red_win:")
print(fights[diff_cols + ["is_red_win"]].corr()["is_red_win"].drop("is_red_win").sort_values())

## Missingness After Cleaning + Feature Engineering\n\n`clean_fights` already handled the source-data gaps (stance, reach, etc). What's left here is mostly *structural* missingness introduced by the history features themselves - a fighter's UFC debut has no prior fights, so `prior_win_rate`/`days_since_last_fight`/`r1_avg_sig_landed_prior` are correctly `NaN`, not 0."

In [ ]:
missing = fights.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(8, max(3, len(missing) * 0.3)))
ax.barh(missing.index, missing.values)
ax.set_xlabel("share missing")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

missing

## Train / Val / Test Drift\n\nBecause the split is chronological, val and test are by construction a different era than train (val: 2020-2023, test: 2023-2026). Checking whether key features have actually shifted matters here - large drift means the model will be evaluated on a meaningfully different distribution than it trained on, which is realistic but should be a known, not a surprise."

In [ ]:
drift_cols = ["experience_diff", "prior_win_rate_diff", "age_diff", "r_days_since_last_fight"]

summary = pd.DataFrame({
    "train": train[drift_cols].mean(),
    "val": val[drift_cols].mean(),
    "test": test[drift_cols].mean(),
})
print(summary)

fig, axes = plt.subplots(1, len(drift_cols), figsize=(18, 4))
for i, col in enumerate(drift_cols):
    axes[i].hist(train[col].dropna(), bins=30, alpha=0.5, label="train", density=True)
    axes[i].hist(val[col].dropna(), bins=30, alpha=0.5, label="val", density=True)
    axes[i].hist(test[col].dropna(), bins=30, alpha=0.5, label="test", density=True)
    axes[i].set_title(col)
axes[0].legend()
plt.tight_layout()
plt.show()

print("\nweight_class share by split (top 5):")
print(pd.DataFrame({
    "train": train["weight_class"].value_counts(normalize=True).head(5),
    "val": val["weight_class"].value_counts(normalize=True).head(5),
    "test": test["weight_class"].value_counts(normalize=True).head(5),
}))

## Correlation Between Engineered Features\n\nChecking for redundant features before they go into a model - e.g. height_diff and reach_diff are likely correlated since taller fighters tend to have longer reach."

In [ ]:
corr_cols = ["height_diff", "reach_diff", "weight_diff", "age_diff",
             "experience_diff", "prior_win_rate_diff", "is_red_win"]
corr = fights[corr_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)
fig.colorbar(im)
plt.tight_layout()
plt.show()

corr